<a href="https://www.kaggle.com/code/samithsachidanandan/neural-network-and-data-loading-with-jax?scriptVersionId=307427706" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Import Libraries

In [1]:
import jax.numpy as jnp
from jax import grad, jit, vmap 
from jax import random 

# Hyperparameters 

Let's get a few bookkeeping items out of the way 

In [2]:
# A helper function to randomly initialize weights and biases 
# for a dense neural network layer 
def random_layer_params(m, n, key, scale=1e-2):
    w_key, b_key = random.split(key)
    return (scale * random.normal(w_key, (n,m)),
           scale * random.normal(b_key, (n,)))


In [3]:
# Initialize all layers 
def init_network_params(sizes, key):
    keys = random.split(key, len(sizes))
    return [random_layer_params(m, n, k)
            for m, n, k in zip(sizes[:-1], sizes[1:], keys )]


layer_sizes = [784, 512, 512, 10]
param_scale = 0.1
step_size = 0.0001
num_epochs = 10
batch_size = 128 
n_targers = 10
params = init_network_params(layer_sizes, random.PRNGKey(0))

In [4]:
for W,b in params:
    print(f'Weights: {W.shape}, bias: {b.shape}')

Weights: (512, 784), bias: (512,)
Weights: (512, 512), bias: (512,)
Weights: (10, 512), bias: (10,)


# Auto-Batching predictions

In [5]:
from jax.scipy.special import logsumexp

def relu(x):
    return jnp.maximum(0,x)

def predict(params, image):
    # per-examples predctions 
    activations = image 
    for w, b in params[:-1]:
        outputs = jnp.dot(w, activations) + b
        activations = relu(outputs)

    final_w, final_b = params[-1]
    logits = jnp.dot(final_w, activations) + final_b
    return logits - logsumexp(logits)

Let's check that our prediction function only works on single images. 

In [6]:
# This works on single examples 

random_flattened_image = random.normal(
    random.PRNGKey(1), (28 * 28,))
preds = predict(params, random_flattened_image)
print(preds.shape)

(10,)


In [7]:
# Doesn't work with a batch 


random_flattened_images = random.normal(
    random.PRNGKey(1), (10, 28 * 28,))
try:
    preds = predict(params, random_flattened_images)
except TypeError:
     print('Invalid shapes')

Invalid shapes


In [8]:
# Make a batched version of the 'predict' function 
batched_predict = vmap(predict, in_axes = (None, 0))

batched_preds = batched_predict(params, random_flattened_images)

print(batched_preds.shape)

(10, 10)


# Utility and loss functions 

In [9]:
def one_hot(x, k, dtype=jnp.float32):
  """Create a one-hot encoding of x of size k."""
  return jnp.array(x[:, None] == jnp.arange(k), dtype)

def accuracy(params, images, targets):
  target_class = jnp.argmax(targets, axis=1)
  predicted_class = jnp.argmax(batched_predict(params, images), axis=1)
  return jnp.mean(predicted_class == target_class)

def loss(params, images, targets):
  preds = batched_predict(params, images)
  return -jnp.mean(preds * targets)

def update(params, x, y):
  grads = grad(loss)(params, x, y)
  return [(w - step_size * dw, b - step_size * db)
          for (w, b), (dw, db) in zip(params, grads)]

# Data loading with tensorflow/datasets

In [10]:
import tensorflow as tf
tf.config.set_visible_devices([], device_type='GPU')

import tensorflow_datasets as tfds


data_dir = '/kaggle/working/tfds'

mnist_data, info = tfds.load(
    name="mnist",
    batch_size=-1,
    data_dir=data_dir,
    with_info=True
)

mnist_data = tfds.as_numpy(mnist_data)
train_data, test_data = mnist_data['train'], mnist_data['test']

num_labels = info.features['label'].num_classes
h, w, c = info.features['image'].shape
num_pixels = h * w * c

# Train
train_images, train_labels = train_data['image'], train_data['label']
train_images = jnp.reshape(train_images, (len(train_images), num_pixels))
train_images = train_images.astype(jnp.float32) / 255.0
train_labels = one_hot(train_labels, num_labels)

# Test
test_images, test_labels = test_data['image'], test_data['label']
test_images = jnp.reshape(test_images, (len(test_images), num_pixels))
test_images = test_images.astype(jnp.float32) / 255.0
test_labels = one_hot(test_labels, num_labels)

2026-03-29 12:34:27.880147: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774787668.115468      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774787668.182519      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774787668.764196      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774787668.764254      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774787668.764257      17 computation_placer.cc:177] computation placer alr

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /kaggle/working/tfds/mnist/incomplete.0SO8LN_3.0.1/mnist-train.tfrecord*...:   0%|          | 0/6000…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /kaggle/working/tfds/mnist/incomplete.0SO8LN_3.0.1/mnist-test.tfrecord*...:   0%|          | 0/10000…

Dataset mnist downloaded and prepared to /kaggle/working/tfds/mnist/3.0.1. Subsequent calls will reuse this data.


In [11]:
print('Train:', train_images.shape, train_labels.shape)
print('Test:', test_images.shape, test_labels.shape)

Train: (60000, 784) (60000, 10)
Test: (10000, 784) (10000, 10)


# Training Loop

In [12]:
import time

def get_train_batches():
  # as_supervised=True gives us the (image, label) as a tuple instead of a dict
  ds = tfds.load(name='mnist', split='train', as_supervised=True, data_dir=data_dir)
  # You can build up an arbitrary tf.data input pipeline
  ds = ds.batch(batch_size).prefetch(1)
  # tfds.dataset_as_numpy converts the tf.data.Dataset into an iterable of NumPy arrays
  return tfds.as_numpy(ds)

for epoch in range(num_epochs):
  start_time = time.time()
  for x, y in get_train_batches():
    x = jnp.reshape(x, (len(x), num_pixels))
    y = one_hot(y, num_labels)
    params = update(params, x, y)
  epoch_time = time.time() - start_time

  train_acc = accuracy(params, train_images, train_labels)
  test_acc = accuracy(params, test_images, test_labels)
  print("Epoch {} in {:0.2f} sec".format(epoch, epoch_time))
  print("Training set accuracy {}".format(train_acc))
  print("Test set accuracy {}".format(test_acc))

Epoch 0 in 18.89 sec
Training set accuracy 0.11243333667516708
Test set accuracy 0.11289999634027481
Epoch 1 in 15.19 sec
Training set accuracy 0.1672833412885666
Test set accuracy 0.16949999332427979
Epoch 2 in 15.07 sec
Training set accuracy 0.21248333156108856
Test set accuracy 0.21649999916553497
Epoch 3 in 15.02 sec
Training set accuracy 0.2615833282470703
Test set accuracy 0.2712000012397766
Epoch 4 in 15.41 sec
Training set accuracy 0.31148332357406616
Test set accuracy 0.3262999951839447
Epoch 5 in 15.07 sec
Training set accuracy 0.35126668214797974
Test set accuracy 0.3658999800682068
Epoch 6 in 15.09 sec
Training set accuracy 0.3810499906539917
Test set accuracy 0.3959999978542328
Epoch 7 in 15.14 sec
Training set accuracy 0.4034999907016754
Test set accuracy 0.41830000281333923
Epoch 8 in 15.07 sec
Training set accuracy 0.42116665840148926
Test set accuracy 0.43479999899864197
Epoch 9 in 15.40 sec
Training set accuracy 0.43488332629203796
Test set accuracy 0.4497999846935272

# Speed up training loop 

In [13]:
@jit
def jit_update(params, x, y):
  grads = grad(loss)(params, x, y)
  return [(w - step_size * dw, b - step_size * db)
          for (w, b), (dw, db) in zip(params, grads)]

In [14]:
params = init_network_params(layer_sizes, random.PRNGKey(0))

In [15]:
for epoch in range(num_epochs):
  start_time = time.time()
  for x, y in get_train_batches():
    x = jnp.reshape(x, (len(x), num_pixels))
    y = one_hot(y, num_labels)
    params = jit_update(params, x, y)
  epoch_time = time.time() - start_time

  train_acc = accuracy(params, train_images, train_labels)
  test_acc = accuracy(params, test_images, test_labels)
  print("Epoch {} in {:0.2f} sec".format(epoch, epoch_time))
  print("Training set accuracy {}".format(train_acc))
  print("Test set accuracy {}".format(test_acc))

Epoch 0 in 5.67 sec
Training set accuracy 0.11243333667516708
Test set accuracy 0.11289999634027481
Epoch 1 in 5.27 sec
Training set accuracy 0.1672833412885666
Test set accuracy 0.16949999332427979
Epoch 2 in 5.02 sec
Training set accuracy 0.21250000596046448
Test set accuracy 0.21649999916553497
Epoch 3 in 5.25 sec
Training set accuracy 0.26161667704582214
Test set accuracy 0.2712000012397766
Epoch 4 in 5.12 sec
Training set accuracy 0.31148332357406616
Test set accuracy 0.3262999951839447
Epoch 5 in 5.22 sec
Training set accuracy 0.35126668214797974
Test set accuracy 0.3658999800682068
Epoch 6 in 5.24 sec
Training set accuracy 0.3810499906539917
Test set accuracy 0.3959999978542328
Epoch 7 in 5.09 sec
Training set accuracy 0.4034999907016754
Test set accuracy 0.41830000281333923
Epoch 8 in 5.30 sec
Training set accuracy 0.42116665840148926
Test set accuracy 0.43479999899864197
Epoch 9 in 5.24 sec
Training set accuracy 0.43488332629203796
Test set accuracy 0.4497999846935272
